In [ ]:
# import os
# import sys
#
# requirements_path = os.path.expanduser("~/Documents/GitHub/CRYPTO_MACAQUES/SECONDARY/requirements.txt")
# !{sys.executable} -m pip install -r "{requirements_path}"

In [ ]:
import os
import sys
import time
!{sys.executable} --version
if sys.version_info.minor == 8:
    raise RuntimeError('USE JUPYTER KERNEL VENV 3.10/310/DEFAULT INSTEAD')

!cd /workspace/CRYPTO_MACAQUES && pip install .
!cd /home/crypto/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && source venv/bin/activate && !{sys.executable} -m pip install .

from IPython.core.display_functions import clear_output
clear_output()

!sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))

In [ ]:
%load_ext autoreload
%autoreload

from dateutil import parser
from SRC.LIBRARIES.new_data_utils import fetch

symbol = 'BTCUSDT'
market_type = 'SPOT'
discretization = '1D'
start_dt = parser.parse('2026-02-01')

df = fetch(market_type, symbol, discretization, start_dt)
print(df.head())

In [ ]:
import ta
#TODO: Write here code to generate MRI indicator over dataframe and add code below to draw that
#Process iteration over timeseries dataframe with sliding window approach
window_size = 30

def stochastic_tradingview(high, low, close, periodK=14, smoothK=3, periodD=3):
    lowest_low = low.rolling(window=periodK).min()
    highest_high = high.rolling(window=periodK).max()
    raw_k = 100 * (close - lowest_low) / (highest_high - lowest_low)
    stoch_k = raw_k.rolling(window=smoothK).mean()
    stoch_d = stoch_k.rolling(window=periodD).mean()

    return stoch_k, stoch_d

df['rsi'] = ta.momentum.RSIIndicator(close=df['close'], window=14).rsi()

df['stoch_k'], df['stoch_d'] = stochastic_tradingview(
    high=df['high'],
    low=df['low'],
    close=df['close'])

macd = ta.trend.MACD(
    close=df['close'],
    window_slow=26,
    window_fast=12,
    window_sign=9)

df['macd_line'] = macd.macd()
df['macd_signal'] = macd.macd_signal()
df['macd_histogram'] = macd.macd_diff()

df['atr'] = ta.volatility.AverageTrueRange(
    high=df['high'],
    low=df['low'],
    close=df['close'],
    window=14).average_true_range()

for i in range(len(df) - window_size + 1):
    window = df.iloc[i:i + window_size]
    # process window
    print(f"{window.index[0]} - {window.index[-1]}")

In [ ]:
%load_ext autoreload
%autoreload

import plotly.graph_objects as go
from plotly.subplots import make_subplots

candlestick_row = 1
volume_row = 2
rsi_row = 3
stoch_row = 4
macd_row = 5
atr_row = 6

def add_bars(col, name, color, row):
    fig.add_trace(
        go.Bar(
            x=df.index,
            y=df[col],
            name=name,
            marker=dict(color=color),
            width=(df.index[1] - df.index[0]).total_seconds() * 1000,
        ),
        row=row, col=1
)

def add_scatter(col, name, color, row, fill=None, fillcolor=None):
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df[col],
            name=name,
            line=dict(color=color, width=2),
            mode='lines',
            fill=fill,
            fillcolor=fillcolor
        ),
        row=row, col=1
    )

def add_over_zone(y0, y1, fillcolor, row):
    fig.add_hrect(
        y0=y0, y1=y1,
        line_width=0,
        fillcolor=fillcolor,
        opacity=0.2,
        row=row, col=1
    )

def add_central_line(row, y=50, line_dash="dot"):
    fig.add_hline(
        y=y,
        line_dash=line_dash,
        line_color="white",
        line_width=1,
        opacity=0.3,
        row=row, col=1
    )

def add_over_zones_and_a_central_line(row):
    add_over_zone(80, 100, "red", row)
    add_over_zone(0, 20, "green", row)
    add_central_line(row)

def add_stoch(speed, color):
    add_scatter('stoch_' + speed, "Stoch %" + speed.capitalize(), color, stoch_row)

fig = make_subplots(
    rows=6, cols=1,
    shared_xaxes=True,
    row_heights=[8, 1, 3, 3, 3, 2],
    vertical_spacing=0.03
)

fig.add_trace(
    go.Candlestick(
        x=df.index,
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="OHLC"
    ),
    row=candlestick_row, col=1
)

add_bars(
    "volume",
    "Volume",
    ["green" if c > o else "red" for o, c in zip(df["open"], df["close"])],
    volume_row)

add_scatter('rsi', "RSI", 'purple', rsi_row)
add_over_zones_and_a_central_line(rsi_row)

add_stoch('k', "lightblue")
add_stoch('d', "orange")
add_over_zones_and_a_central_line(stoch_row)

prev_macd = df['macd_histogram'].shift(1)
macd_colors = [
    'green' if (val >= 0 and (i == 0 or val >= prev_macd.iloc[i]))
    else 'lightgreen' if (val >= 0 and val < prev_macd.iloc[i])
    else 'red' if (val < 0 and (i == 0 or val <= prev_macd.iloc[i]))
    else 'lightcoral' if (val < 0 and val > prev_macd.iloc[i])
    else 'rgba(128, 128, 128, 0.3)'
    for i, val in enumerate(df['macd_histogram'])
]
add_scatter("macd_line", "MACD Line", 'lightblue', macd_row)
add_scatter("macd_signal", "Signal Line", 'orange', macd_row)
add_bars("macd_histogram", "MACD Histogram", macd_colors, macd_row)
add_central_line(macd_row, line_dash="solid")

add_scatter('atr', "ATR (14)", 'orange', atr_row, 'tozeroy', 'rgba(255, 165, 0, 0.1)')
add_central_line(atr_row, y=df['atr'].mean())

fig.update_layout(bargap=0)

fig.update_layout(
    title="OHLC with Volume Explosion After Valley",
    xaxis_rangeslider_visible=False,
    yaxis_title="Price",
    yaxis2_title="Volume",
    yaxis3_title="RSI",
    yaxis4_title="Stoch",
    yaxis5_title="MACD",
    yaxis6_title="ATR",
    height=1700
)

fig.show()